# LiteLLM: One Unified API for Every LLM Provider
## One API, Every Model — Provider-Agnostic Agent Design
⏱ ~12 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esturban/agent/blob/main/examples/65-litellm-multi-provider/litellm_workbook.ipynb)

**LiteLLM** is an open-source proxy library that gives every LLM provider — OpenAI, Anthropic, Google Gemini, Cohere, Mistral, and 100+ more — a **single, identical Python interface**. You write your code once; swapping providers means changing one string.

By the end of this workshop you will understand *why* a unified API matters, *how* LiteLLM routes calls to each provider, and *how* to add fallback chains, cost tracking, and LangChain integration to any project.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is LiteLLM and why does it exist? |
| 2 | **Unified Completion** — Same call, every provider |
| 3 | **Fallback Chains** — Automatic retry on a backup model |
| 4 | **Cost Tracking** — Per-call USD cost with `completion_cost()` |
| 5 | **Load Balancing** — Spread requests across models or API keys |
| 6 | **LangChain Integration** — Drop-in `ChatLiteLLM` replacement |
| 7 | **Streaming** — Real-time token streaming across providers |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with requirements installed, or Google Colab)
- `OPENAI_API_KEY` — **required** for all cells
- `ANTHROPIC_API_KEY` — optional (provider sections skip gracefully if missing)
- `GEMINI_API_KEY` — optional
- `COHERE_API_KEY` — optional

### Key References
> LiteLLM documentation: https://docs.litellm.ai  
> Supported providers: https://docs.litellm.ai/docs/providers  
> BerriAI/litellm on GitHub: https://github.com/BerriAI/litellm  
> LMSYS Chatbot Arena (model benchmark leaderboard): https://huggingface.co/spaces/lmsys/chatbot-arena-leaderboard  
> Artificial Analysis (independent LLM benchmark): https://artificialanalysis.ai

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "litellm",
            "langchain-community",
            "langchain-core",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API keys ─────────────────────────────────────────────────────────────────
# Colab:  set in Secrets panel (left sidebar key icon)
# Local:  create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    for _optional in ("ANTHROPIC_API_KEY", "GEMINI_API_KEY", "COHERE_API_KEY"):
        try:
            os.environ[_optional] = userdata.get(_optional)
        except Exception:
            pass  # optional keys — skip silently
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ─────────────────────────────────────────────────────────────
openai_key = os.environ.get("OPENAI_API_KEY", "")
assert openai_key.startswith("sk-"), (
    "OPENAI_API_KEY is missing or invalid.\n"
    "Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)

print("API keys loaded. Provider availability:")
for provider, env_var in [
    ("OpenAI    ", "OPENAI_API_KEY"),
    ("Anthropic ", "ANTHROPIC_API_KEY"),
    ("Gemini    ", "GEMINI_API_KEY"),
    ("Cohere    ", "COHERE_API_KEY"),
]:
    status = "READY" if os.environ.get(env_var) else "skipped (key not set)"
    print(f"  {provider}: {status}")

---

## Part 1 — What Is LiteLLM and Why Does It Exist?

---

### The Problem: Provider Lock-In

Every LLM provider ships its own Python SDK with its own API shape:

| Provider | Import | Call shape |
|----------|--------|------------|
| OpenAI | `openai` | `client.chat.completions.create(model=..., messages=...)` |
| Anthropic | `anthropic` | `client.messages.create(model=..., max_tokens=..., messages=...)` |
| Google Gemini | `google.generativeai` | `model.generate_content(contents=...)` |
| Cohere | `cohere` | `co.chat(model=..., message=...)` |
| Mistral | `mistralai` | `client.chat.complete(model=..., messages=...)` |

If you write your app against OpenAI and want to try Anthropic, you rewrite every call site. Add fallback logic on top and the code becomes a maze of `if provider == ...` branches.

**LiteLLM solves this** by implementing the OpenAI API shape for every provider. You call `litellm.completion()` with any model string and LiteLLM handles the translation internally.

---

### LiteLLM vs Writing One SDK Per Provider

| Concern | Per-provider SDKs | LiteLLM |
|---------|------------------|---------|
| API surface to learn | One per provider | One (`completion()`) |
| Swapping providers | Rewrite call sites | Change model string |
| Fallback on error | Manual try/except | `Router` with `fallbacks=` |
| Cost tracking | Manual token math | `completion_cost()` built in |
| LangChain compatibility | Separate wrappers | `ChatLiteLLM` drop-in |
| Streaming | Provider-specific | Unified `stream=True` |

---

### Architecture Diagram

```
YOUR CODE
──────────────────────────────────────────────────────────────

  litellm.completion(
      model="openai/gpt-4o-mini",   ← or "anthropic/claude-haiku-4-5"
      messages=[{"role": "user", "content": "..."}],
  )

         │  identical call every time
         ▼

  ┌────────────────────────────────────────────┐
  │          LiteLLM Unified API Layer         │
  │                                            │
  │  • Routes based on model string prefix     │
  │  • Translates request to provider format   │
  │  • Normalises response to OpenAI shape     │
  │  • Tracks token usage + computes cost      │
  │  • Router: fallbacks, retries, load bal.   │
  └────────────┬───────────────────────────────┘
               │
     ┌─────────┼──────────┬──────────┬──────────┐
     ▼         ▼          ▼          ▼          ▼
 ┌───────┐ ┌───────┐ ┌───────┐ ┌───────┐ ┌───────┐
 │OpenAI │ │Anthro-│ │Gemini │ │Cohere │ │Mistral│
 │  API  │ │  pic  │ │  API  │ │  API  │ │  API  │
 └───┬───┘ └───┬───┘ └───┬───┘ └───┬───┘ └───┬───┘
     └─────────┴──────────┴──────────┴──────────┘
               │
               ▼
  Unified response object (always OpenAI shape)
  response.choices[0].message.content
```

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Model string** | Provider prefix + model name, e.g. `"anthropic/claude-haiku-4-5"` |
| **Router** | LiteLLM component that handles fallbacks, retries, and load balancing |
| **Fallback** | A backup model tried automatically when the primary fails |
| **completion_cost()** | Built-in function returning USD cost for any LiteLLM response |
| **ChatLiteLLM** | LangChain `BaseChatModel` backed by LiteLLM — drop-in for `ChatOpenAI` |
| **Stream** | Returning tokens incrementally as they are generated |
| **Provider prefix** | The part before `/` in the model string — tells LiteLLM which SDK to use |

---

## Part 2 — Unified Completion: Same Call, Every Provider

---

### How the Model String Works

The model string follows the pattern `"<provider>/<model-name>"`. LiteLLM reads the prefix to decide which SDK and endpoint to use:

```
"openai/gpt-4o-mini"           → routes to OpenAI Chat Completions API
"anthropic/claude-haiku-4-5"  → routes to Anthropic Messages API
"gemini/gemini-1.5-flash"      → routes to Google Generative AI API
"cohere/command-r"             → routes to Cohere Chat API
"ollama/llama3"                → routes to local Ollama server (no API key)
```

The response object is always the same shape — `response.choices[0].message.content` — regardless of which provider answered.

In [ ]:
# ─── 2-A: Imports and shared prompt ──────────────────────────────────────────
import litellm
from litellm import completion, completion_cost

litellm.set_verbose = False  # suppress debug output; set True to see raw requests

# A single prompt we'll send to every provider — makes comparison easy.
PROMPT = "Explain neural networks in exactly one sentence."

print("LiteLLM version:", litellm.__version__)
print("Shared prompt:  ", PROMPT)

In [ ]:
# ─── 2-B: OpenAI — always available ──────────────────────────────────────────
# This is the baseline call. Notice: identical signature to openai SDK,
# but via litellm.completion() instead of client.chat.completions.create().

response_oai = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=80,
)

cost_oai = completion_cost(completion_response=response_oai)

print("Provider:  openai/gpt-4o-mini")
print(f"Cost:      ${cost_oai:.6f}")
print(f"Tokens:    {response_oai.usage.total_tokens} total")
print(f"Answer:    {response_oai.choices[0].message.content}")

In [ ]:
# ─── 2-C: Anthropic, Gemini, Cohere — same call, optional keys ───────────────
# If a key is not set, the cell prints a skip message showing what the
# model string *would* be — so you can see the pattern without running the call.

optional_providers = [
    ("ANTHROPIC_API_KEY", "anthropic/claude-haiku-4-5"),
    ("GEMINI_API_KEY",    "gemini/gemini-1.5-flash"),
    ("COHERE_API_KEY",    "cohere/command-r"),
]

for env_var, model_str in optional_providers:
    if os.environ.get(env_var):
        resp = completion(
            model=model_str,
            messages=[{"role": "user", "content": PROMPT}],
            max_tokens=80,
        )
        cost = completion_cost(completion_response=resp)
        print(f"[{model_str}]")
        print(f"  Cost:   ${cost:.6f}")
        print(f"  Answer: {resp.choices[0].message.content}")
    else:
        print(f"[{model_str}]")
        print(f"  Skipped — {env_var} not set")
        print(f"  To enable: add {env_var}=... to your .env or Colab Secrets")
    print()

---

## Part 3 — Fallback Chains: Automatic Retry on Backup Models

---

### Why Fallbacks Matter

In production, LLM APIs fail. Common causes:
- **Rate limits** (HTTP 429) — you've exceeded requests-per-minute
- **Service outages** — provider is down or degraded
- **Context length exceeded** — input is too long for the chosen model
- **Quota exceeded** — your account has used its monthly allocation

Without fallbacks, a single rate limit crashes your application. With LiteLLM's `Router`, failure is invisible to the caller — the router retries automatically on the next model in the list.

---

### Fallback Strategies

| Strategy | How it works | Best for |
|----------|-------------|----------|
| **Model fallback** | On error, try next model in list | Provider outages, rate limits |
| **API key rotation** | Same model, different API keys | Staying under per-key rate limits |
| **Context-length fallback** | If input too long, route to larger context model | Variable-length inputs |
| **Cost-threshold fallback** | Estimate cost first; downgrade if over budget | Cost-controlled pipelines |

---

### Router Fallback Flow

```
router.completion(model="primary", messages=[...])
        │
        ▼
  ┌─────────────┐
  │ Try primary │  gpt-4o-mini  ← attempt 1
  └──────┬──────┘
         │  429 Rate Limit  or  500 Server Error
         ▼
  ┌──────────────────┐
  │ Try fallback-1   │  anthropic/claude-haiku-4-5  ← attempt 2
  └──────┬───────────┘
         │  still failing
         ▼
  ┌──────────────────┐
  │ Try fallback-2   │  cohere/command-r  ← attempt 3
  └──────────────────┘
         │  success
         ▼
  Response returned to caller — caller never saw the errors
```

In [ ]:
# ─── 3-A: Build a Router with fallbacks ──────────────────────────────────────
from litellm import Router

# model_list defines every model available to the router.
# Each entry has a 'model_name' alias (used in router.completion() calls)
# and 'litellm_params' which map to litellm.completion() keyword args.
model_list = [
    {
        "model_name": "primary",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
    {
        "model_name": "fallback-cheap",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",  # same model, different slot
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
]

# Add Anthropic fallback only if the key is available
if os.environ.get("ANTHROPIC_API_KEY"):
    model_list.append(
        {
            "model_name": "fallback-anthropic",
            "litellm_params": {
                "model": "anthropic/claude-haiku-4-5",
                "api_key": os.environ.get("ANTHROPIC_API_KEY"),
            },
        }
    )

router = Router(
    model_list=model_list,
    fallbacks=[{"primary": ["fallback-cheap", "fallback-anthropic"]}],
    num_retries=2,   # retry same model before falling back
    retry_after=1,   # seconds to wait between retries
)

print(f"Router ready with {len(model_list)} model entries")
for entry in model_list:
    print(f"  alias={entry['model_name']:20} model={entry['litellm_params']['model']}")

In [ ]:
# ─── 3-B: Normal call through the router ─────────────────────────────────────
# When the primary model succeeds, the caller sees a normal response.
# The fallback chain is invisible.

router_response = router.completion(
    model="primary",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=80,
)

print(f"Model used:  {router_response.model}")
print(f"Answer:      {router_response.choices[0].message.content}")
print()
print("If 'primary' had returned a 429 or 500 error, the router would have")
print("silently retried on 'fallback-cheap', then 'fallback-anthropic'.")

In [ ]:
# ─── 3-C: Simulate a fallback by making the primary fail ─────────────────────
# We trigger a forced failure by providing an invalid API key on the primary.
# The router catches the AuthenticationError and moves to working-fallback.

failing_model_list = [
    {
        "model_name": "broken-primary",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": "sk-INVALID-KEY-WILL-FAIL",  # intentionally wrong
        },
    },
    {
        "model_name": "working-fallback",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),  # real key
        },
    },
]

fallback_demo_router = Router(
    model_list=failing_model_list,
    fallbacks=[{"broken-primary": ["working-fallback"]}],
    num_retries=0,  # go straight to fallback on first failure
)

print("Calling 'broken-primary' (invalid key) — router should fall to 'working-fallback'...")
try:
    fb_response = fallback_demo_router.completion(
        model="broken-primary",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=60,
    )
    print(f"Router succeeded via fallback. Model used: {fb_response.model}")
    print(f"Answer: {fb_response.choices[0].message.content}")
except Exception as exc:
    print(f"All models failed: {type(exc).__name__}: {exc}")

---

## Part 4 — Cost Tracking: Per-Call USD Cost

---

### How LiteLLM Tracks Cost

`completion_cost()` reads the token usage from any LiteLLM response and looks up the price per token from LiteLLM's built-in pricing database (updated with each library release). It returns a `float` in USD.

```python
response = completion(model="openai/gpt-4o-mini", messages=[...])
cost = completion_cost(completion_response=response)
# cost = 0.000012  (USD)
```

You can also estimate cost *before* making a call:

```python
input_cost, output_cost = litellm.cost_per_token(
    model="openai/gpt-4o",
    prompt_tokens=100,
    completion_tokens=50,
)
```

---

### Provider Price Comparison (approximate, mid-2025)

| Provider | Model | Input per 1M tokens | Output per 1M tokens | Speed tier |
|----------|-------|--------------------|--------------------|------------|
| OpenAI | gpt-4o-mini | $0.15 | $0.60 | Fast |
| OpenAI | gpt-4o | $2.50 | $10.00 | Medium |
| Anthropic | claude-haiku-4-5 | $0.25 | $1.25 | Fast |
| Anthropic | claude-sonnet-4-5 | $3.00 | $15.00 | Medium |
| Google | gemini-1.5-flash | $0.075 | $0.30 | Fast |
| Cohere | command-r | $0.15 | $0.60 | Medium |
| Ollama | any local model | $0.00 | $0.00 | Hardware-limited |

> Always verify current prices at each provider's pricing page — these change frequently.

In [ ]:
# ─── 4-A: Cost vs quality across OpenAI models ───────────────────────────────
# Same prompt, two models at different price points.
# This is the most common trade-off decision in production.

openai_models = [
    "openai/gpt-4o-mini",
    "openai/gpt-4o",
]

detailed_prompt = (
    "You are a senior software engineer. In 2-3 sentences, explain the trade-offs "
    "between using a microservices architecture versus a monolith for a startup."
)

print(f"Prompt: {detailed_prompt[:80]}...\n")
print(f"{'Model':<28} {'Input tok':>10} {'Output tok':>11} {'Cost USD':>12}")
print("-" * 65)

for model in openai_models:
    resp = completion(
        model=model,
        messages=[{"role": "user", "content": detailed_prompt}],
        max_tokens=120,
    )
    cost = completion_cost(completion_response=resp)
    print(
        f"{model:<28}"
        f"{resp.usage.prompt_tokens:>10}"
        f"{resp.usage.completion_tokens:>11}"
        f"  ${cost:>10.6f}"
    )

print()
print("gpt-4o costs ~16x more per output token than gpt-4o-mini.")
print("For most tasks, gpt-4o-mini quality is sufficient — cost matters at scale.")

In [ ]:
# ─── 4-B: Accumulate cost across a simulated batch of requests ───────────────
# Realistic scenario: process a list of questions and track cumulative spend.

questions = [
    "What is the capital of France?",
    "Summarise the concept of recursion in one sentence.",
    "What does HTTP stand for?",
    "Name the three primary colours.",
    "In one sentence, what is a database index?",
]

total_cost = 0.0
total_input_tokens = 0
total_output_tokens = 0

print(f"Processing {len(questions)} questions with openai/gpt-4o-mini...\n")
for i, q in enumerate(questions, 1):
    resp = completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": q}],
        max_tokens=60,
    )
    cost = completion_cost(completion_response=resp)
    total_cost += cost
    total_input_tokens += resp.usage.prompt_tokens
    total_output_tokens += resp.usage.completion_tokens
    print(f"  [{i}] ${cost:.6f}  {resp.choices[0].message.content[:70]}")

print()
print("Batch summary:")
print(f"  Total input tokens:  {total_input_tokens}")
print(f"  Total output tokens: {total_output_tokens}")
print(f"  Total cost:          ${total_cost:.6f}")
print(f"  Average per call:    ${total_cost / len(questions):.6f}")

---

## Part 5 — Load Balancing: Spread Requests Across Models or Keys

---

### Why Load Balance?

A single API key has a requests-per-minute (RPM) cap. For high-throughput pipelines a single key hits its rate limit and returns 429 errors. With multiple keys or multiple providers in the Router, requests are distributed across all entries, keeping each under its cap.

### Load Balancing Strategies in LiteLLM Router

| Strategy | Description | When to use |
|----------|-------------|-------------|
| `simple-shuffle` | Random selection from the model list | Default; good for most cases |
| `least-busy` | Pick the model with fewest in-flight requests | Low-latency pipelines |
| `latency-based-routing` | Track p95 latency, route to fastest | Latency-sensitive apps |
| `usage-based-routing` | Track token usage, route to least-used | Staying under rate limits |

In [ ]:
# ─── 5-A: Round-robin load balancing across two API key slots ─────────────────
# In production: put different API keys in each slot to double your RPM cap.
# Here we use the same key twice to demonstrate the routing pattern.

load_balanced_models = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
    {
        "model_name": "gpt-pool",  # same alias = both treated as one pool
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),  # would be key #2 in prod
        },
    },
]

lb_router = Router(
    model_list=load_balanced_models,
    routing_strategy="simple-shuffle",
)

print("Sending 4 requests through the load-balanced pool...\n")
for i in range(4):
    resp = lb_router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say the number {i + 1} only."}],
        max_tokens=5,
    )
    print(f"  Request {i + 1}: model={resp.model}  answer={resp.choices[0].message.content.strip()}")

print()
print("In production: put two different OPENAI_API_KEY values in the two slots.")
print("Each key handles ~50% of traffic — doubling your effective RPM cap.")

---

## Part 6 — LangChain Integration via `ChatLiteLLM`

---

### Why `ChatLiteLLM`?

If your project uses LangChain chains, agents, or LangGraph nodes, you are likely already using `ChatOpenAI`. `ChatLiteLLM` is a **drop-in replacement** — it implements the same `BaseChatModel` interface. Change one import and one constructor call; everything else stays the same.

```python
# Before — locked to OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

# After — any provider via LiteLLM
from langchain_community.chat_models import ChatLiteLLM
llm = ChatLiteLLM(model="anthropic/claude-haiku-4-5")
```

The chain, prompt, and output parser are **unchanged**.

In [ ]:
# ─── 6-A: ChatLiteLLM as a standalone chat model ─────────────────────────────
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatLiteLLM(model="openai/gpt-4o-mini", max_tokens=80)

messages = [
    SystemMessage(content="You are a helpful assistant that responds concisely."),
    HumanMessage(content="What are the three pillars of object-oriented programming?"),
]

response = chat.invoke(messages)
print("ChatLiteLLM response:")
print(response.content)
print()
print("To switch to Anthropic, change the model string — nothing else:")
print('  ChatLiteLLM(model="anthropic/claude-haiku-4-5")')

In [ ]:
# ─── 6-B: ChatLiteLLM inside a LangChain LCEL chain ─────────────────────────
# A full chain (prompt | llm | parser) where the LLM can be ANY provider,
# hot-swappable at runtime without touching the chain definition.

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

SYSTEM = (
    "You are a technical glossary. When given a term, respond with:\n"
    "1. A one-sentence definition\n"
    "2. One real-world example\n"
    "Keep your total response under 60 words."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM),
        ("human", "Term: {term}"),
    ]
)

# Swap this model string to change providers — chain is identical
llm = ChatLiteLLM(model="openai/gpt-4o-mini", max_tokens=100)

chain = prompt | llm | StrOutputParser()

terms = ["idempotency", "eventual consistency", "backpressure"]
for term in terms:
    answer = chain.invoke({"term": term})
    print(f"--- {term} ---")
    print(answer)
    print()

In [ ]:
# ─── 6-C: Provider-agnostic factory function ─────────────────────────────────
# Real-world pattern: configure the LLM via environment variable.
# The rest of the codebase never references a specific provider.


def get_llm(model_env_var: str = "LLM_MODEL", default: str = "openai/gpt-4o-mini"):
    """Return a ChatLiteLLM instance using the model from an env var.

    Set LLM_MODEL=anthropic/claude-haiku-4-5 in .env to switch providers
    without changing any application code.
    """
    model = os.environ.get(model_env_var, default)
    print(f"  [get_llm] Using model: {model}")
    return ChatLiteLLM(model=model, max_tokens=60)


# Simulate switching provider via environment variable
os.environ["LLM_MODEL"] = "openai/gpt-4o-mini"

factory_llm = get_llm()
factory_chain = prompt | factory_llm | StrOutputParser()
result = factory_chain.invoke({"term": "memoisation"})
print(result)
print()
print("To switch provider: os.environ['LLM_MODEL'] = 'anthropic/claude-haiku-4-5'")
print("Then re-run this cell — no other code changes needed.")

---

## Part 7 — Streaming: Real-Time Token Output

---

### Why Stream?

By default, `completion()` waits until the model finishes generating before returning. For long outputs this means the user sees nothing for several seconds. With `stream=True`, tokens are yielded as they arrive — the same mechanism used by the ChatGPT web UI.

```python
# Without streaming — blocks until complete
response = completion(model="...", messages=[...])   # waits 3-5s
print(response.choices[0].message.content)            # prints all at once

# With streaming — tokens arrive incrementally
for chunk in completion(model="...", messages=[...], stream=True):
    token = chunk.choices[0].delta.content or ""
    print(token, end="", flush=True)                  # prints as generated
```

LiteLLM normalises streaming across all providers — the chunk structure is always the same regardless of which backend is answering.

In [ ]:
# ─── 7-A: Streaming a response token by token ────────────────────────────────

stream_prompt = (
    "Write a short (4-5 sentence) explanation of how HTTPS works, "
    "suitable for a non-technical audience."
)

print("Streaming response (tokens arrive as generated):")
print("-" * 60)

stream = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": stream_prompt}],
    max_tokens=150,
    stream=True,
)

full_text = ""
for chunk in stream:
    token = chunk.choices[0].delta.content or ""
    full_text += token
    print(token, end="", flush=True)

print()
print("-" * 60)
print(f"Total characters received: {len(full_text)}")

In [ ]:
# ─── 7-B: Streaming with a LangChain LCEL chain ──────────────────────────────
# LCEL chains support .stream() — each yielded item is a plain string chunk.

streaming_llm = ChatLiteLLM(model="openai/gpt-4o-mini", max_tokens=150)

simple_prompt = ChatPromptTemplate.from_messages(
    [("human", "List 5 benefits of test-driven development, one per line.")]
)

streaming_chain = simple_prompt | streaming_llm | StrOutputParser()

print("LangChain streaming output:")
print("-" * 60)
for chunk in streaming_chain.stream({}):
    print(chunk, end="", flush=True)
print()
print("-" * 60)
print("Same pattern works with 'anthropic/claude-haiku-4-5' or any other model.")

---

## Exercises

---

### Exercise 1 — Add a New Provider to the Cost Comparison

Extend the cost comparison table from Part 4 to include Cohere (`cohere/command-r`).

- If `COHERE_API_KEY` is set, make the call and print cost + answer.
- If the key is not set, print a skip message that shows what the model string would be.

**Goal:** The loop should work identically regardless of how many keys are set — no `if provider == "cohere"` in the caller, just a clean data-driven loop.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

COMPARISON_PROMPT = "In one sentence, what is the difference between a list and a tuple in Python?"

# TODO: define a list of (env_var, model_string) tuples that includes
#       openai/gpt-4o-mini, openai/gpt-4o, and cohere/command-r

ALL_PROVIDERS = [
    # ("ENV_VAR", "provider/model"),
    # TODO: fill this in
]

print(f"{'Model':<32} {'Cost':>12}  Answer / Status")
print("-" * 80)

for env_var, model_str in ALL_PROVIDERS:
    if os.environ.get(env_var):
        # TODO: call completion(), compute cost, print result
        pass
    else:
        print(f"{model_str:<32} {'skipped':>12}  {env_var} not set")

---

### Exercise 2 — Cost-Threshold Fallback

Write a function `smart_completion(prompt, budget_usd)` that:

1. Estimates the cost of calling `openai/gpt-4o` using `litellm.cost_per_token()` *before* making the call.
2. If the estimated cost exceeds `budget_usd`, falls back to `openai/gpt-4o-mini` instead.
3. Makes the actual call with whichever model was selected and returns the answer string.

**Hint:** `litellm.cost_per_token(model, prompt_tokens, completion_tokens)` returns `(input_cost, output_cost)`. Use `len(prompt.split())` as a rough token estimate.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────


def smart_completion(prompt: str, budget_usd: float) -> str:
    """Call the best model that fits within budget_usd.

    Preferred: openai/gpt-4o
    Fallback:  openai/gpt-4o-mini (if estimated cost exceeds budget)
    """
    preferred_model = "openai/gpt-4o"
    fallback_model = "openai/gpt-4o-mini"

    # TODO: estimate prompt tokens (use len(prompt.split()) as approximation)
    # TODO: call litellm.cost_per_token() to estimate input cost
    # TODO: choose model based on estimated cost vs budget_usd
    # TODO: call completion() and return answer string

    pass


# Test calls — uncomment after implementing
# print("Tight budget (forces fallback):")
# print(smart_completion(PROMPT, budget_usd=0.000001))
# print()
# print("Generous budget (uses preferred):")
# print(smart_completion(PROMPT, budget_usd=1.00))

---

### Exercise 3 — Multi-Provider Comparison Runner

Write a function `compare_providers(prompt, providers)` that:

1. Accepts a prompt string and a list of `(env_var, model_string)` tuples.
2. Calls each provider that has its key set.
3. Returns a list of dicts with keys: `model`, `answer`, `cost_usd`, `latency_ms`.
4. Prints a summary table sorted by cost (cheapest first).

**Why this matters:** In production you often run the same prompt across two or three models during evaluation to decide which to deploy. This function automates that workflow.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
import time


def compare_providers(prompt: str, providers: list) -> list:
    """Run prompt across all available providers and return comparison results."""
    results = []

    for env_var, model_str in providers:
        if not os.environ.get(env_var):
            continue

        # TODO: record start time with time.perf_counter()
        # TODO: call completion()
        # TODO: compute latency_ms = (perf_counter() - t0) * 1000
        # TODO: compute cost with completion_cost()
        # TODO: append dict with model, answer, cost_usd, latency_ms to results

    return results


eval_prompt = "Explain the CAP theorem in two sentences."
eval_providers = [
    ("OPENAI_API_KEY",    "openai/gpt-4o-mini"),
    ("OPENAI_API_KEY",    "openai/gpt-4o"),
    ("ANTHROPIC_API_KEY", "anthropic/claude-haiku-4-5"),
    ("GEMINI_API_KEY",    "gemini/gemini-1.5-flash"),
]

# Uncomment after implementing compare_providers:
# comparison = compare_providers(eval_prompt, eval_providers)
# comparison.sort(key=lambda r: r["cost_usd"])
# print(f"{'Model':<35} {'Cost':>10} {'Latency':>10}")
# print("-" * 60)
# for r in comparison:
#     print(f"{r['model']:<35} ${r['cost_usd']:>9.6f} {r['latency_ms']:>8.0f}ms")

---

## What's Next?

You now have a complete foundation in LiteLLM's unified API, fallback chains, cost tracking, load balancing, LangChain integration, and streaming. Here's where to go deeper:

### Integrate with agents and RAG
- **Example 12 — RAG with ChromaDB** ([`../12-basic-rag-notebook/rag_workbook.ipynb`](../12-basic-rag-notebook/rag_workbook.ipynb)): swap `ChatOpenAI` for `ChatLiteLLM` in the LCEL chain — your entire RAG pipeline becomes provider-agnostic.
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): multi-step RAG agent where the LLM layer can be any provider via `ChatLiteLLM`.

### Production hardening
- **LiteLLM Proxy Server**: run LiteLLM as a standalone OpenAI-compatible proxy so any application that speaks the OpenAI API can use it — https://docs.litellm.ai/docs/proxy/quick_start
- **Logging callbacks**: `litellm.success_callback` and `litellm.failure_callback` send every request/response to Langfuse, Helicone, or your own endpoint.
- **Budget controls**: `litellm.max_budget` and per-key budget alerts for multi-user or multi-team deployments.

### Advanced routing
- **Latency-based routing**: route to whichever model is currently fastest (`routing_strategy="latency-based-routing"`).
- **Context-length fallback**: automatically upgrade to a larger-context model when input exceeds the primary model's window.
- **Prompt caching**: Anthropic and OpenAI both support prompt caching — LiteLLM surfaces cache hit/miss in `response.usage`.

### Key references
> LiteLLM full provider list: https://docs.litellm.ai/docs/providers  
> LiteLLM Router docs: https://docs.litellm.ai/docs/routing  
> LMSYS Chatbot Arena leaderboard: https://huggingface.co/spaces/lmsys/chatbot-arena-leaderboard  
> Artificial Analysis (speed, cost, and quality benchmark): https://artificialanalysis.ai  
> BerriAI/litellm GitHub: https://github.com/BerriAI/litellm

---

*Workshop complete. The logical next step is Example 12 — add this multi-provider layer to a production RAG pipeline.*

---

## Exercise Answer Key

Attempt the exercises yourself before reading these. These are sample solutions — not the only correct answers.

### Answer 1 — Add Cohere to the Cost Comparison

The key design insight: keep the provider list as data, not as branching code. A single loop handles every provider — those with keys get called, those without print a skip message. Adding a new provider means adding one tuple to the list and nothing else.

In [ ]:
# Answer 1 — Extended provider cost comparison
COMPARISON_PROMPT = "In one sentence, what is the difference between a list and a tuple in Python?"

ALL_PROVIDERS = [
    ("OPENAI_API_KEY",    "openai/gpt-4o-mini"),
    ("OPENAI_API_KEY",    "openai/gpt-4o"),
    ("ANTHROPIC_API_KEY", "anthropic/claude-haiku-4-5"),
    ("GEMINI_API_KEY",    "gemini/gemini-1.5-flash"),
    ("COHERE_API_KEY",    "cohere/command-r"),
]

print(f"{'Model':<35} {'Cost USD':>12}  Answer / Status")
print("-" * 85)

for env_var, model_str in ALL_PROVIDERS:
    if os.environ.get(env_var):
        resp = completion(
            model=model_str,
            messages=[{"role": "user", "content": COMPARISON_PROMPT}],
            max_tokens=60,
        )
        cost = completion_cost(completion_response=resp)
        answer_preview = resp.choices[0].message.content[:55].replace("\n", " ")
        print(f"{model_str:<35} ${cost:>10.6f}  {answer_preview}")
    else:
        print(f"{model_str:<35} {'skipped':>12}  {env_var} not set")

### Answer 2 — Cost-Threshold Fallback

The key insight: estimate cost *before* making the expensive call using `cost_per_token()`. This avoids calling gpt-4o just to discover it was over budget. The `split()` token estimate is approximate — production code uses `tiktoken` for precision. The function prints the decision reasoning so it is auditable.

In [ ]:
# Answer 2 — Cost-threshold fallback


def smart_completion(prompt: str, budget_usd: float) -> str:
    """Call the best model that fits within budget_usd.

    Preferred: openai/gpt-4o
    Fallback:  openai/gpt-4o-mini (if estimated cost exceeds budget)
    """
    preferred_model = "openai/gpt-4o"
    fallback_model = "openai/gpt-4o-mini"
    assumed_output_tokens = 60  # conservative estimate for short answers

    # Rough token estimate: words * 1.3 is a practical approximation
    estimated_input_tokens = int(len(prompt.split()) * 1.3)

    input_cost, output_cost = litellm.cost_per_token(
        model=preferred_model,
        prompt_tokens=estimated_input_tokens,
        completion_tokens=assumed_output_tokens,
    )
    estimated_total = input_cost + output_cost

    if estimated_total <= budget_usd:
        chosen_model = preferred_model
        reason = f"estimated ${estimated_total:.6f} fits within ${budget_usd:.6f} budget"
    else:
        chosen_model = fallback_model
        reason = f"estimated ${estimated_total:.6f} exceeds ${budget_usd:.6f} — downgraded"

    print(f"  Model selected: {chosen_model}")
    print(f"  Reason:         {reason}")

    resp = completion(
        model=chosen_model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=assumed_output_tokens,
    )
    return resp.choices[0].message.content


print("=== Tight budget (expected: forces fallback to gpt-4o-mini) ===")
answer_tight = smart_completion(PROMPT, budget_usd=0.0000001)
print(f"  Answer: {answer_tight}")
print()
print("=== Generous budget (expected: uses preferred gpt-4o) ===")
answer_generous = smart_completion(PROMPT, budget_usd=1.00)
print(f"  Answer: {answer_generous}")

### Answer 3 — Multi-Provider Comparison Runner

The function uses `time.perf_counter()` for millisecond-precision latency measurement and returns structured dicts that are easy to sort, filter, log, or feed into an evaluation harness. The caller controls sorting — the function just collects data. This separation of concerns makes it reusable in CI pipelines.

In [ ]:
# Answer 3 — Multi-provider comparison runner
import time


def compare_providers(prompt: str, providers: list) -> list:
    """Run prompt across all available providers and return comparison results."""
    results = []

    for env_var, model_str in providers:
        if not os.environ.get(env_var):
            print(f"  Skipping {model_str} — {env_var} not set")
            continue

        print(f"  Calling {model_str}...")
        t0 = time.perf_counter()
        resp = completion(
            model=model_str,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=80,
        )
        latency_ms = (time.perf_counter() - t0) * 1000
        cost = completion_cost(completion_response=resp)

        results.append(
            {
                "model": model_str,
                "answer": resp.choices[0].message.content,
                "cost_usd": cost,
                "latency_ms": latency_ms,
                "total_tokens": resp.usage.total_tokens,
            }
        )

    return results


eval_prompt = "Explain the CAP theorem in two sentences."
eval_providers = [
    ("OPENAI_API_KEY",    "openai/gpt-4o-mini"),
    ("OPENAI_API_KEY",    "openai/gpt-4o"),
    ("ANTHROPIC_API_KEY", "anthropic/claude-haiku-4-5"),
    ("GEMINI_API_KEY",    "gemini/gemini-1.5-flash"),
]

print(f"Running comparison for: '{eval_prompt[:60]}'\n")
comparison = compare_providers(eval_prompt, eval_providers)
comparison.sort(key=lambda r: r["cost_usd"])  # cheapest first

print()
print(f"{'Model':<35} {'Cost USD':>12} {'Latency':>10} {'Tokens':>7}")
print("-" * 68)
for r in comparison:
    print(
        f"{r['model']:<35} ${r['cost_usd']:>10.6f}"
        f" {r['latency_ms']:>8.0f}ms"
        f" {r['total_tokens']:>6}"
    )

print()
if comparison:
    cheapest = comparison[0]
    print(f"Cheapest: {cheapest['model']} at ${cheapest['cost_usd']:.6f}")
    print(f"Answer:   {cheapest['answer'][:120]}")